# Prepare stimuli_videos.csv for annotation

Reads `data/stimuli_videos.csv` (hand-annotated salient feature manifest) and adds three columns
needed by `annotate_videos.py`:

| column | type | description |
|--------|------|-------------|
| `video_path` | str | absolute path to the matching `_stripped.mp4` in `stimuli/main_blocks/` |
| `start_time_s` | float | `time` column converted from `M:SS` to seconds |
| `end_time_s` | float | start of the next row (same block) or video duration (last row of each block) |

Run this notebook **once** before running `annotate_videos.py`.  
It overwrites `data/stimuli_videos.csv` in-place and is safe to re-run (idempotent).

In [ ]:
import os
import sys
import cv2
import pandas as pd
from pathlib import Path

## Configuration

The notebook auto-detects the repo root by searching upward for the `data/` directory.
If it can't find it, set `REPO_ROOT` manually below.

In [ ]:
# Auto-detect repo root: walk upward until we find the 'data/' directory.
# When running from preprocessing/segmentation/, this resolves two levels up.
_here = Path(os.getcwd())
REPO_ROOT = _here
while not (REPO_ROOT / "data").is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# ── Edit these if auto-detection fails ────────────────────────────────────────
# REPO_ROOT = Path("/labs/vislearnlab/experiments/movie-watching")

CSV_PATH    = REPO_ROOT / "data" / "stimuli_videos.csv"
STIMULI_DIR = REPO_ROOT / "stimuli" / "main_blocks"

print(f"Repo root   : {REPO_ROOT}")
print(f"CSV path    : {CSV_PATH}")
print(f"Stimuli dir : {STIMULI_DIR}")

assert CSV_PATH.exists(),      f"CSV not found: {CSV_PATH}"
assert STIMULI_DIR.is_dir(),   f"Stimuli dir not found: {STIMULI_DIR}"

## Load CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH.name}")
print(f"Columns    : {df.columns.tolist()}")
print(f"Video blocks: {df['video_block'].unique().tolist()}")
df.head()

## Map video block labels → video file paths

The `video_block` names in the CSV (e.g. `"Sesame US 1"`) don't directly match the
filenames in `stimuli/main_blocks/` (e.g. `sesameus_1_stripped.mp4`).  
The dict below handles every block, including the typo `"Franck Complex"` in the CSV.

In [ ]:
# Maps each video_block label to the filename stem in stimuli/main_blocks/.
# 'Franck Complex' is a typo in the CSV; the file is 'frank_complex'.
BLOCK_TO_STEM: dict[str, str] = {
    "Sesame US 1":    "sesameus_1",
    "Sesame US 2":    "sesameus_2",
    "Sesame India 1": "sesameindia_1",
    "Sesame India 2": "sesameindia_2",
    "Slow Animals":   "slow_animals",
    "Slow Hands":     "slow_hands",
    "Slow Inanimate": "slow_inanimate",
    "Slow People":    "slow_people",
    "Frank Complex": "frank_complex",   
    "Frank Objects":  "frank_objects",
    "Frank Play":     "frank_play",
    "Pixar Birds":    "pixar_birds",
}

# Check for any blocks in the CSV that are not in the mapping
unknown_blocks = set(df["video_block"].unique()) - set(BLOCK_TO_STEM)
if unknown_blocks:
    print(f"WARNING — no mapping for these blocks (add them to BLOCK_TO_STEM above): {unknown_blocks}")
else:
    print("All video blocks have a mapping")

In [ ]:
def block_to_video_path(block: str) -> str | None:
    """
    Return the absolute path to the _stripped.mp4 for `block`,
    falling back to the non-stripped version if needed.
    Returns None if the block is not in BLOCK_TO_STEM.
    """
    stem = BLOCK_TO_STEM.get(block)
    if stem is None:
        return None
    # Prefer audio-stripped versions — they're the same content but smaller
    stripped  = STIMULI_DIR / f"{stem}_stripped.mp4"
    fallback  = STIMULI_DIR / f"{stem}.mp4"
    chosen    = stripped if stripped.exists() else fallback
    return str(chosen)


df["video_path"] = df["video_block"].apply(block_to_video_path)

# Report which paths were found
print("Video path resolution:")
for block, path in df.groupby("video_block")["video_path"].first().items():
    exists = path is not None and Path(path).exists()
    name   = Path(path).name if path else "MISSING"
    print(f" {block:20s}  →  {name}")

n_missing = df["video_path"].isna().sum()
if n_missing:
    print(f"\nWARNING: {n_missing} rows have no video path")

## Convert `time` column from M:SS to seconds

In [ ]:
def time_to_seconds(t: str) -> float:
    """Parse 'M:SS' or 'MM:SS' string into float seconds (e.g. '1:23' → 83.0)."""
    t = str(t).strip()
    parts = t.split(":")
    if len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    # Fallback: treat as raw seconds
    return float(parts[0])


df["start_time_s"] = df["time"].apply(time_to_seconds)

print("Sample time conversions (first 15 rows):")
print(df[["video_block", "time", "start_time_s"]].head(15).to_string(index=True))

## Read video durations

Needed to set `end_time_s` for the last row of each video block.

In [ ]:
# Cache duration for each unique video path using OpenCV.
video_duration_s: dict[str, float] = {}

print("Video durations:")
for vp in sorted(df["video_path"].dropna().unique()):
    cap     = cv2.VideoCapture(vp)
    fps     = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    dur = n_frames / fps if fps > 0 else 0.0
    video_duration_s[vp] = dur
    print(f"  {Path(vp).name:45s}  {dur:7.2f}s  {fps:.2f} fps  {n_frames:5d} frames")

## Compute `end_time_s` per row

Rule:
- If the **next** row belongs to the same `video_block`, `end_time_s = next_row.start_time_s`
- Otherwise (last row of a block), `end_time_s = video duration`

In [ ]:
end_times: list[float] = []

for i in range(len(df)):
    row = df.iloc[i]
    next_is_same_block = (
        i + 1 < len(df)
        and df.iloc[i + 1]["video_block"] == row["video_block"]
    )
    if next_is_same_block:
        # End at the start of the next annotation row
        end_t = df.iloc[i + 1]["start_time_s"]
    else:
        # Last row of this block — use the actual video duration
        vp    = row["video_path"]
        end_t = video_duration_s.get(vp, row["start_time_s"] + 5.0)
    end_times.append(round(end_t, 4))

df["end_time_s"] = end_times

# Sanity check — every chunk must span a positive duration
bad = df[df["end_time_s"] <= df["start_time_s"]]
if not bad.empty:
    print(f"WARNING: {len(bad)} rows with zero or negative duration:")
    print(bad[["video_block", "time", "start_time_s", "end_time_s"]].to_string())
else:
    print(f"All {len(df)} chunks have positive duration")

df[["video_block", "time", "start_time_s", "end_time_s", "salient_variable"]].head(25)

## Summary statistics

In [ ]:
df["chunk_duration_s"] = df["end_time_s"] - df["start_time_s"]

print("Chunk duration statistics (seconds):")
print(df["chunk_duration_s"].describe().round(2).to_string())
print()

# Count prompts per row
df["n_prompts"] = df["salient_variable"].apply(
    lambda v: len([p for p in str(v).split(",") if p.strip()])
)
total_jobs = df["n_prompts"].sum()
print(f"Total annotation jobs (rows × prompts): {total_jobs}")
print(f"  avg prompts/row: {df['n_prompts'].mean():.1f}")
print()

print("Per-block summary:")
print(
    df.groupby("video_block")[["chunk_duration_s", "n_prompts"]]
    .agg({"chunk_duration_s": ["count", "sum", "mean"], "n_prompts": "sum"})
    .round(2)
    .to_string()
)

## Save enriched CSV

Overwrites `data/stimuli_videos.csv` in-place with the new columns added.
The `chunk_duration_s` and `n_prompts` helper columns are dropped before saving —
they can be recomputed at any time and are only for QC above.

In [ ]:
# Drop the temporary helper columns we added for summary stats
save_df = df.drop(columns=["chunk_duration_s", "n_prompts"], errors="ignore")

save_df.to_csv(CSV_PATH, index=False)
print(f"Saved {len(save_df)} rows → {CSV_PATH}")
print(f"Final columns: {save_df.columns.tolist()}")

## Generate stimuli_prompts_round0.csv

Expands each row of `stimuli_videos.csv` into one row per individual annotation prompt, applies a prompt transformation (e.g. `face (monster)` → `monster face`), and writes `data/stimuli_prompts.csv`.

This file is the primary input to `annotate_videos.py`.

Two optional columns, `include_coord` and `exclude_coord`, can be filled in manually after generation (format: `"x,y"` in pixel coordinates).  Leave blank to use text-only prompts.

In [ ]:
import re


def split_annotations(salient_var: str) -> list[str]:
    """
    Split a comma-separated salient_variable string into individual annotations,
    respecting commas that appear inside parentheses.

    e.g. "face (child, man), hands"  →  ["face (child, man)", "hands"]
    """
    results = []
    current: list[str] = []
    depth = 0
    for ch in str(salient_var):
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == "," and depth == 0:
            token = "".join(current).strip()
            if token:
                results.append(token)
            current = []
        else:
            current.append(ch)
    token = "".join(current).strip()
    if token:
        results.append(token)
    return results


def transform_prompt(annotation: str) -> str:
    """
    Move parenthetical qualifier to the front so SAM 3 gets the most specific
    noun first.

    Examples:
        "face (monster)"    → "monster face"
        "face (child, man)" → "child, man face"
        "hands"             → "hands"
    """
    annotation = annotation.strip()
    m = re.match(r"^(.+?)\s*\((.+?)\)\s*$", annotation)
    if m:
        base      = m.group(1).strip()
        qualifier = m.group(2).strip()
        return f"{qualifier} {base}"
    return annotation


# Quick smoke-test
_tests = [
    ("face (cow)",        "cow face"),
    ("face (child, man)", "child, man face"),
    ("hands",             "hands"),
    ("faces (kids)",      "kids faces"),
]
for _raw, _expected in _tests:
    _got = transform_prompt(_raw)
    assert _got == _expected, f"transform_prompt({_raw!r}) = {_got!r}, expected {_expected!r}"

print("transform_prompt: all smoke-tests passed")

In [ ]:
PROMPTS_PATH = REPO_ROOT / "data" / "stimuli_prompts.csv"

# Re-read the enriched stimuli_videos.csv (in case this cell is run standalone)
sv = pd.read_csv(CSV_PATH)

prompt_rows = []
prompt_idx  = 0

for row_idx, row in sv.iterrows():
    annotations = split_annotations(row["salient_variable"])
    for ann in annotations:
        prompt_rows.append(dict(
            prompt_idx       = prompt_idx,
            row_idx          = row_idx,
            video_block      = row["video_block"],
            time             = row["time"],
            video_path       = row["video_path"],
            start_time_s     = row["start_time_s"],
            end_time_s       = row["end_time_s"],
            annotation_raw   = ann,
            prompt           = transform_prompt(ann),
            include_coord    = "",   # fill in manually to add a foreground point
            exclude_coord    = "",   # fill in manually to add a background point
        ))
        prompt_idx += 1

prompts_df = pd.DataFrame(prompt_rows)

print(f"Expanded {len(sv)} rows → {len(prompts_df)} prompt rows")
print(f"Columns: {prompts_df.columns.tolist()}")
prompts_df.head(10)

In [ ]:
prompts_df.to_csv(PROMPTS_PATH, index=False)
print(f"Saved {len(prompts_df)} rows → {PROMPTS_PATH}")

# Quick sanity check: every row_idx in stimuli_videos.csv is represented
missing = set(sv.index) - set(prompts_df["row_idx"].unique())
if missing:
    print(f"WARNING: {len(missing)} row_idx values have no prompts: {missing}")
else:
    print(f"All {sv.index.nunique()} row_idx values covered.")